# 039_scd2
Silver load for a type 2 history (`store_history`). The flow's script returns the latest
snapshot of each member; `apply_scd2` in `001_nb_utils` compares it with the current versions
and closes or opens versions (new, changed and removed members). `primary_key_columns` is the
business key.

In [ ]:
# Bootstrap: find the pipeline root and load the shared notebooks.
# run_notebook() plays the part of Databricks' %run: it runs another notebook's code cells in
# this notebook's namespace. It is used instead of Jupyter's %run because that breaks on
# Windows paths with spaces.
import json
import pathlib

PIPELINE_ROOT = next(
    folder
    for folder in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
    if (folder / "00_shared").is_dir()
)


def run_notebook(relative_path):
    notebook = json.loads((PIPELINE_ROOT / relative_path).read_text(encoding="utf-8"))
    for cell in notebook["cells"]:
        if cell["cell_type"] == "code":
            exec("".join(cell["source"]), globals())


if "SHARED_LOADED" not in globals():
    run_notebook("00_shared/000_nb_configurations.ipynb")
    run_notebook("00_shared/001_nb_utils.ipynb")
    SHARED_LOADED = True

## Loader

In [ ]:
def silver_scd2(config, batch_id):
    if config.get("full_refresh"):
        print(f"  {config['destination_table']}: full refresh ignored, SCD2 history is kept")
    return apply_scd2(
        run_script(config),
        "silver",
        config["destination_schema"],
        config["destination_table"],
        config["primary_key_columns"],
        batch_id,
    )


LOADERS["039_scd2"] = silver_scd2

## Run on its own

In [ ]:
# Parameters: set a control_id and run all to load one flow on its own (handy for testing).
# Left at 0, this notebook only defines its loader for the orchestrator.
control_id = 0
batch_id = ""

In [ ]:
if control_id:
    print(execute_flow(control_id, batch_id or new_batch_id(), silver_scd2, triggered_by="manual"))